In [3]:
import pandas as pd
import numpy as np
import json

In [1]:
PATH_DATA = "data/"

In [5]:
df_icd = pd.read_csv(PATH_DATA +"LIBCIM10MULTI.TXT", sep="|",header=None,names=["code","aut_mco","pos","aut_ssr","lib_court","libelle"],encoding="latin-1")
df_icd.code = df_icd.code.str.replace(" ","")
df_index_icd = pd.read_csv(PATH_DATA + "cim_index_modifie.csv", sep=";")

In [44]:
df_icd_chap20 = pd.read_csv(PATH_DATA +"LIBCIM10MULTI_ch20.TXT", sep="|",header=None,names=["code","aut_mco","pos","aut_ssr","lib_court","libelle"],encoding="latin-1")
df_icd_chap20.code = df_icd_chap20.code.str.replace(" ","")

cat_motif = ["Z"+ str(x).zfill(2) for x in range(0,55)]
cat_facteurs = ["Z"+ str(x).zfill(2) for x in range(55,100)]
cat_sympt = ["R"+ str(x).zfill(2) for x in range(0,100)]

In [45]:
df_icd= df_icd.assign(groupe = np.where(df_icd.code.isin(df_icd_chap20.code),"Causes externes",
                             np.where(df_icd.code.str.slice(0,3).isin(cat_motif),"Motifs de recours",
                             np.where(df_icd.code.str.slice(0,3).isin(cat_facteurs),"Facteurs influents",
                             np.where(df_icd.code.str.slice(0,3).isin(cat_sympt),"Symptomes",
                             "Diagnostics")))))

In [46]:
df_index_icd = df_index_icd.assign(groupe = np.where(df_index_icd.code.isin(df_icd_chap20.code),"Causes externes",
                             np.where(df_index_icd.code.str.slice(0,3).isin(cat_motif),"Motifs de recours",
                             np.where(df_index_icd.code.str.slice(0,3).isin(cat_facteurs),"Facteurs influents",
                             np.where(df_index_icd.code.str.slice(0,3).isin(cat_sympt),"Symptomes",
                             "Diagnostics")))))

In [47]:
df_index_icd[df_index_icd.code.str.slice(-1) == "9" ].drop_duplicates("code").groupby("groupe").size()

groupe
Diagnostics           914
Facteurs influents     17
Motifs de recours      35
Symptomes              19
dtype: int64

In [48]:
df_index_icd[(df_index_icd.code.str.slice(-1) == "9") & ( df_index_icd.groupe=="Facteurs influents")  ]

,code,icd_description,index_orginal,index_reformulate,groupe
275362,Z579,Difficultés liées à l'exposition professionnel...,"Difficulté(s) de(s), liées à, exposition à, fa...",difficultés liées à l'exposition à un facteur ...,Facteurs influents
275363,Z579,Difficultés liées à l'exposition professionnel...,"Difficulté(s) de(s), liées à, exposition à, fa...",difficultés liées à l'exposition professionnel...,Facteurs influents
275364,Z579,Difficultés liées à l'exposition professionnel...,"Difficulté(s) de(s), liées à, exposition à, fa...",difficultés d'exposition à un facteur de risqu...,Facteurs influents
275365,Z579,Difficultés liées à l'exposition professionnel...,"Difficulté(s) de(s), liées à, exposition à, fa...",difficultés d'exposition professionnelle à un ...,Facteurs influents
275366,Z579,Difficultés liées à l'exposition professionnel...,"Difficulté(s) de(s), liées à, exposition à, fa...",difficulté liée à l'exposition à un facteur de...,Facteurs influents
...,...,...,...,...,...
280607,Z899,"Absence acquise de membre, sans précision","Moignon (chirurgical) d'amputation, guéri ou a...",amputation avec moignon guéri,Facteurs influents
280608,Z899,"Absence acquise de membre, sans précision","Moignon (chirurgical) d'amputation, guéri ou a...",amputation avec moignon ancien,Facteurs influents
280609,Z899,"Absence acquise de membre, sans précision","Moignon (chirurgical) d'amputation, guéri ou a...",amputation avec moignon chirurgical,Facteurs influents
280610,Z899,"Absence acquise de membre, sans précision","Moignon (chirurgical) d'amputation, guéri ou a...",amputation avec moignon chirurgical guéri,Facteurs influents


In [52]:
df_index_icd[(df_index_icd.code.str.slice(-1) == "9") & ( df_index_icd.groupe=="Diagnostics")  ].to_excel("data/Controle_index_sai.xlsx",index=False)

In [37]:
df_icd[df_icd.code.str.slice(-1) == "9" ].drop_duplicates("code").groupby("groupe").size()

groupe
Causes externes       3690
Diagnostics           1539
Facteurs influents      35
Motifs de recours       38
Symptomes               35
dtype: int64

In [10]:
df_icd_chapitres = pd.read_excel(PATH_DATA +"icd_chapters.xlsx")


In [17]:
df_icd = df_icd.assign(categorie = df_icd.code.str.slice(0,3))

In [19]:
df_icd.merge(df_icd_chapitres,left_on="categorie",right_on="diag_deb",how="left").ffill()

,code,aut_mco,pos,aut_ssr,lib_court,libelle,categorie,hiera,diag_deb,diag_fin,lib,annee_deb,annee_fin,en_cours,niveau_fin
0,A00,3,NNN,3,CHOLERA,Choléra,A00,1.0,A00,B99,Certaines maladies infectieuses et parasitaires,2009.0,2025.0,1.0,0.0
1,A000,0,OOO,0,"CHOLERA A VIBRIO CHOLERAE 01, BIOVAR CHOLERAE","Choléra à Vibrio cholerae 01, biovar cholerae",A00,1.0,A00,B99,Certaines maladies infectieuses et parasitaires,2009.0,2025.0,1.0,0.0
2,A001,0,OOO,0,"CHOLERA A VIBRIO CHOLERAE 01, BIOVAR EL TOR","Choléra à Vibrio cholerae 01, biovar El Tor",A00,1.0,A00,B99,Certaines maladies infectieuses et parasitaires,2009.0,2025.0,1.0,0.0
3,A009,0,OOO,0,"CHOLERA, SAI","Choléra, sans précision",A00,1.0,A00,B99,Certaines maladies infectieuses et parasitaires,2009.0,2025.0,1.0,0.0
4,A01,3,NNN,3,FIEVRES TYPHOIDE ET PARATYPHOIDE,Fièvres typhoïde et paratyphoïde,A01,1.0,A00,B99,Certaines maladies infectieuses et parasitaires,2009.0,2025.0,1.0,0.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
42892,Z992+8,0,ONO,0,"DEPENDANCE ENVERS UNE DIALYSE RENALE, NCA","Dépendance envers une dialyse rénale, autre",Z99,21.0,Z00,Z80,Facteurs influant sur l'état de santé et motif...,2009.0,2025.0,1.0,0.0
42893,Z993,0,ONO,0,DEPENDANCE ENVERS UN FAUTEUIL ROULANT,Dépendance envers un fauteuil roulant,Z99,21.0,Z00,Z80,Facteurs influant sur l'état de santé et motif...,2009.0,2025.0,1.0,0.0
42894,Z994,0,ONO,0,DEPENDANCE ENVERS UN COEUR ARTIFICIEL,Dépendance envers un coeur artificiel,Z99,21.0,Z00,Z80,Facteurs influant sur l'état de santé et motif...,2009.0,2025.0,1.0,0.0
42895,Z998,0,ONO,0,DEPENDANCE ENVERS D'AUTRES MACHINES ET APP. AU...,Dépendance envers d'autres machines et apparei...,Z99,21.0,Z00,Z80,Facteurs influant sur l'état de santé et motif...,2009.0,2025.0,1.0,0.0


In [53]:
df_icd[df_icd.code.str.contains('N02')]

,code,aut_mco,pos,aut_ssr,lib_court,libelle,categorie,groupe
10445,N02,3,NNN,3,"HEMATURIE RECID., PERSIST.",Hématurie récidivante et persistante,N02,Diagnostics
10446,N020,0,OOO,0,"HEMATURIE RECID., PERSIST. AVEC ANOM. GLOM. MI...",Hématurie récidivante et persistante avec anom...,N02,Diagnostics
10447,N0200,0,OOO,0,"HEMATURIE RECID., PERSIST. AVEC ANOM. GLOM. MI...",Hématurie récidivante et persistante avec anom...,N02,Diagnostics
10448,N0209,0,OOO,0,"HEMATURIE RECID., PERSIST. AVEC ANOM. GLOM. MI...",Hématurie récidivante et persistante avec anom...,N02,Diagnostics
10449,N021,0,OOO,0,"HEMATURIE RECID., PERSIST. AVEC LES. GLOM. SEG...",Hématurie récidivante et persistante avec lési...,N02,Diagnostics
10450,N0210,0,OOO,0,"HEMATURIE RECID., PERSIST. AVEC LES. GLOM. SEG...",Hématurie récidivante et persistante avec lési...,N02,Diagnostics
10451,N0219,0,OOO,0,"HEMATURIE RECID., PERSIST. AVEC LES. GLOM. SEG...",Hématurie récidivante et persistante avec lési...,N02,Diagnostics
10452,N022,0,OOO,0,"HEMATURIE RECID., PERSIST. AVEC G.N. MEMBRANEU...",Hématurie récidivante et persistante avec glom...,N02,Diagnostics
10453,N023,0,OOO,0,"HEMATURIE RECID., PERSIST. AVEC G.N. PROLIF. M...",Hématurie récidivante et persistante avec glom...,N02,Diagnostics
10454,N024,0,OOO,0,"HEMATURIE RECID., PERSIST. AVEC G.N. PROLIF. E...",Hématurie récidivante et persistante avec glom...,N02,Diagnostics


In [55]:
df_index_icd[df_index_icd.index_reformulate=="hématurie"]

,code,icd_description,index_orginal,index_reformulate,groupe
144982,N029,"Hématurie récidivante et persistante, sans pré...","Hématurie (essentielle), intermittente (voir a...",hématurie,Diagnostics
144986,N029,"Hématurie récidivante et persistante, sans pré...","Hématurie (essentielle), paroxystique (voir au...",hématurie,Diagnostics
228862,R31,"Hématurie, sans précision",Hématurie (essentielle),hématurie,Symptomes
228864,R31,"Hématurie, sans précision","Hématurie (essentielle), due aux sulfamides, m...",hématurie,Symptomes
228880,R31,Hématurie,"Hémorragie (de) (due à), urinaire nca",hématurie,Symptomes


In [68]:
df_tmp = df_index_icd.drop_duplicates(["code","index_reformulate"]).groupby("index_reformulate").size().to_frame('nb').reset_index()

In [69]:
df_tmp = df_tmp[df_tmp.nb>1]

In [73]:
df_index_icd[df_index_icd.index_reformulate.isin(df_tmp.index_reformulate)].sort_values("index_reformulate").to_excel("data/duplicates_index_entries.xlsx")